# quantization-mnist-ffnn-pytorch

Two paths to a smaller, faster MNIST classifier — post-training quantization (PTQ) via `nnx.quantize_int8` and quantization-aware training (QAT) via `nnx.qat_train_step_factory` + `nnx.QATLifecycleCallback`. Trade-off study: model size, validation accuracy, and inference latency.


# 1. Overview

## 1.1 Task & motivation

Once a model trains, you'd often like to deploy it smaller and faster. Two recipes:

- **PTQ** (post-training quantization) — quantize an already-trained model in one shot. Cheap (no extra training), but accuracy can drop if the weight distribution doesn't quantize cleanly.
- **QAT** (quantization-aware training) — insert fake-quant ops during training so the optimizer sees the quantization noise and adapts to it. Slower, but typically recovers more accuracy than PTQ at the same bitwidth.

`nnx` ships both via the `torchao` backend: `nnx.quantize_int8` for INT8 weight-only PTQ, and `nnx.qat_train_step_factory` + `nnx.QATLifecycleCallback` for "8da4w" QAT (8-bit dynamic activations, 4-bit weights).

## 1.2 Dataset summary

MNIST handwritten digits via `torchvision.datasets.MNIST`, wrapped in `nnx.NNDataset`. Same constants as the sibling `image_classification-mnist-ffnn-pytorch` task.

## 1.3 Approach in one paragraph

Train a baseline FFN. PTQ it via `quantize_int8`. Separately train a fresh model with the QAT callback + step factory. Compare: validation accuracy of FP32 baseline vs PTQ-int8 vs QAT-8da4w; pickled state-dict size; per-batch inference latency on CPU.

## 1.4 Libraries used

`nnx` (`NNModel`, `NNDataset`, `quantize_int8`, `qat_train_step_factory`, `QATLifecycleCallback`), `torch`, `torchvision`, `torchao` (PTQ + QAT backend; opt-in via `nnx[quantize]`).


# 2. Environment & Setup

## 2.1 Imports

In [1]:
SMOKE_TEST = 0
SMOKE_TEST_EPOCHS = 1


In [2]:
import pickle
import time

import torch
import torchvision as thv
from prettytable import PrettyTable

import nnx
from nnx.nn.dataset.nn_dataset import NNDataset
from nnx.nn.enum.activations import Activations
from nnx.nn.enum.devices import Devices
from nnx.nn.enum.losses import Losses
from nnx.nn.enum.nets import Nets
from nnx.nn.enum.optims import Optims
from nnx.nn.nn_model import NNModel
from nnx.nn.params.nn_model_params import NNModelParams
from nnx.nn.params.nn_optim_params import NNOptimParams
from nnx.nn.params.nn_params import NNParams
from nnx.nn.params.nn_train_params import NNTrainParams


## 2.2 Configuration / hyperparameters

In [3]:
DS_MEAN: float = 0.1307
DS_STD: float = 0.3081

# 8da4w QAT uses a default int4 groupsize of 32 — hidden widths must
# be multiples of 32, otherwise the quantizer needs padding_allowed=True
# and the comparison gets noisy. Pick widths that divide cleanly.
HIDDEN_DIMS = [128, 64]
N_EPOCHS = SMOKE_TEST_EPOCHS if SMOKE_TEST else 3
BATCH_SIZE = 128
LR = 1e-3


## 2.3 Reproducibility (seed, device)

In [4]:
nnx.set_seed(0)
DEVICE = Devices.CPU


# 3. Data

## 3.1 Loading

In [5]:
ds = NNDataset(
    ds_class=thv.datasets.MNIST,
    transform=thv.transforms.Compose([
        thv.transforms.ToTensor(),
        thv.transforms.Normalize(mean=DS_MEAN, std=DS_STD),
    ]),
)
print(f"input_dim={ds.input_dim}, output_dim={ds.output_dim}")


input_dim=784, output_dim=10


## 3.2 Inspection / EDA

Same MNIST as the sibling pytorch-MNIST task — see that notebook's §3 for class-distribution + per-class samples. This notebook reuses the same data wrapper.

## 3.3 Preprocessing & splits

`NNDataset` provides the standard torchvision train/val split. Normalize transform applied above.


# 4. Model

## 4.1 Architecture

A small FP32 FFN baseline. The QAT path will train a fresh copy of the same architecture; the PTQ path will quantize the baseline post-hoc.


In [6]:
def make_model():
    return NNModel(
        net_params=NNParams(
            input_dim=ds.input_dim,
            output_dim=ds.output_dim,
            hidden_dims=HIDDEN_DIMS,
            dropout_prob=0.0,
            activation=Activations.RELU,
        ),
        params=NNModelParams(
            net=Nets.FEED_FWD,
            device=DEVICE,
            loss=Losses.CROSS_ENTROPY,
        ),
    )

def train_params():
    return NNTrainParams(
        n_epochs=N_EPOCHS,
        train_loader=ds.train_loader,
        val_loader=ds.val_loader,
        optim=NNOptimParams(
            name=Optims.ADAM,
            max_lr=LR,
            momentum=(0.9, 0.999),
            weight_decay=0.0,
        ),
    )


## 4.2 PTQ + QAT contracts

- **`nnx.quantize_int8(model)`** returns a *new* `NNModel` whose linear weights are packed `int8` (per-channel scales). Activations stay FP. The contract is "same forward shape, possibly different output values within a quantization tolerance".
- **`nnx.qat_train_step_factory(qat_config="8da4w") + QATLifecycleCallback(qat_config="8da4w")`** swaps the model's linears for fake-quant variants at `on_train_begin` and converts them to truly-quantized variants at `on_train_end`. Pass both into `model.train(...)`.

## 4.3 Why this design

8da4w (8-bit dynamic activation + 4-bit weight) is the dominant edge-deployment recipe for transformer LLMs; the same backend works on FFN classifiers and gives a clean A/B story for PTQ vs QAT at very low bitwidths.


# 5. Training

## 5.1 Train the FP32 baseline

In [7]:
fp32_model = make_model()
fp32_run = fp32_model.train(params=train_params())
fp32_final_val_loss = fp32_run.idps[-1].val_edp.loss
print(f"FP32 baseline: {len(fp32_run.idps)} iterations, final val_loss={fp32_final_val_loss:.4f}")


+--------------------------------------------------------------+
|                        Run Details...                        |
+---------------------------+----------------------------------+
|             id            | e9ae2a0a456a0efb8b06a4241b3ec16f |
|         model.net         |             feed_fwd             |
|         model.loss        |          cross_entropy           |
|        model.device       |               cpu                |
|       net.input_dim       |               784                |
|       net.output_dim      |                10                |
|      net.dropout_prob     |               0.0                |
|      net.hidden_dims      |            [128, 64]             |
|       net.activation      |               relu               |
|       train.n_epochs      |                3                 |
|     train.optim.max_lr    |              0.001               |
|    train.optim.momentum   |           (0.9, 0.999)           |
|      train.optim.name  

Training:   0%|          | 0/3 [00:00<?, ?it/s]

Training:  33%|███▎      | 1/3 [00:01<00:03,  1.89s/it]

Training:  33%|███▎      | 1/3 [00:02<00:03,  1.89s/it, error=0.7892, lr=0.0010]

Training:  67%|██████▋   | 2/3 [00:04<00:02,  2.02s/it, error=0.7892, lr=0.0010]

Training:  67%|██████▋   | 2/3 [00:04<00:02,  2.02s/it, error=0.5688, lr=0.0010]

Training: 100%|██████████| 3/3 [00:05<00:00,  2.00s/it, error=0.5688, lr=0.0010]

Training: 100%|██████████| 3/3 [00:06<00:00,  2.00s/it, error=0.4652, lr=0.0010]

Training: 100%|██████████| 3/3 [00:06<00:00,  2.06s/it, error=0.4652, lr=0.0010]


Run saved to /Users/kaveh/repos/ml-lab/.claude/worktrees/overnight-cleanup/runs/e9ae2a0a456a0efb8b06a4241b3ec16f
FP32 baseline: 3 iterations, final val_loss=2.0587


## 5.2 PTQ — `nnx.quantize_int8`

In [8]:
ptq_model = nnx.quantize_int8(fp32_model)
print(f"PTQ model type: {type(ptq_model).__name__}")


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0).


W0529 20:29:10.021000 84453 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


PTQ model type: NNModel


## 5.3 QAT — `qat_train_step_factory` + `QATLifecycleCallback`

In [9]:
qat_model = make_model()
qat_cb = nnx.QATLifecycleCallback(qat_config="8da4w")
qat_step = nnx.qat_train_step_factory(qat_config="8da4w")

qat_run = qat_model.train(
    params=train_params(),
    callbacks=[qat_cb],
    train_step_fn=qat_step,
)
print(f"QAT callback: is_prepared={qat_cb.is_prepared}, is_converted={qat_cb.is_converted}")
print(f"QAT final val_loss (note: measured on FP-shadow before conversion): {qat_run.idps[-1].val_edp.loss:.4f}")

# Sanity: after train(), the model's linears are the truly-quantized
# variant (not the fake-quant variant); we look them up by class name
# because torchao's converted linear doesn't subclass nn.Linear.
classes_after_qat = {type(m).__name__ for m in qat_model.net.modules()}
print("converted classes containing 'Int':", sorted(c for c in classes_after_qat if "Int" in c))


/Users/kaveh/.pyenv/versions/3.11.0/lib/python3.11/site-packages/torchao/quantization/quant_primitives.py:96: UserWarning: Deprecation: TorchAODType is deprecated, please use the torch.intN dtype instead (e.g. TorchAODType.INT4 -> torch.int4)
  warnings.warn(


+--------------------------------------------------------------+
|                        Run Details...                        |
+---------------------------+----------------------------------+
|             id            | e9ae2a0a456a0efb8b06a4241b3ec16f |
|         model.net         |             feed_fwd             |
|         model.loss        |          cross_entropy           |
|        model.device       |               cpu                |
|       net.input_dim       |               784                |
|       net.output_dim      |                10                |
|      net.dropout_prob     |               0.0                |
|      net.hidden_dims      |            [128, 64]             |
|       net.activation      |               relu               |
|       train.n_epochs      |                3                 |
|     train.optim.max_lr    |              0.001               |
|    train.optim.momentum   |           (0.9, 0.999)           |
|      train.optim.name  

Training:   0%|          | 0/3 [00:00<?, ?it/s]

Training:  33%|███▎      | 1/3 [00:01<00:03,  1.82s/it]

Training:  33%|███▎      | 1/3 [00:02<00:03,  1.82s/it, error=0.7932, lr=0.0010]

...

Training:  67%|██████▋   | 2/3 [00:04<00:01,  1.92s/it, error=0.6827, lr=0.0010]

Training: 100%|██████████| 3/3 [00:05<00:00,  1.99s/it, error=0.6827, lr=0.0010]

Training: 100%|██████████| 3/3 [00:06<00:00,  1.99s/it, error=0.5547, lr=0.0010]

Training: 100%|██████████| 3/3 [00:06<00:00,  2.03s/it, error=0.5547, lr=0.0010]


Run saved to /Users/kaveh/repos/ml-lab/.claude/worktrees/overnight-cleanup/runs/e9ae2a0a456a0efb8b06a4241b3ec16f
QAT callback: is_prepared=True, is_converted=True
QAT final val_loss (note: measured on FP-shadow before conversion): 2.0729
converted classes containing 'Int': ['Int8DynActInt4WeightLinear']


# 6. Evaluation & Results

## 6.1 Accuracy + size + latency table

In [10]:
def eval_loss_acc(model, loader):
    """Compute mean loss + top-1 accuracy without disturbing model state."""
    was_training = model.net.training
    model.net.eval()
    try:
        loss_fn = torch.nn.CrossEntropyLoss()
        total, correct, loss_sum, n_batches = 0, 0, 0.0, 0
        with torch.no_grad():
            for X, y in loader:
                X = X.view(X.size(0), -1).float()
                y = y.long()
                logits = model.net(X)
                loss_sum += loss_fn(logits, y).item()
                n_batches += 1
                preds = logits.argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        return loss_sum / max(1, n_batches), correct / max(1, total)
    finally:
        if was_training:
            model.net.train()

def state_size_bytes(model):
    return len(pickle.dumps(model.net.state_dict()))

def avg_latency_us(model, n_batches=10):
    """Mean per-batch forward latency in microseconds (CPU, eval mode)."""
    model.net.eval()
    it = iter(ds.val_loader)
    # warm up
    X, _ = next(it)
    X = X.view(X.size(0), -1).float()
    with torch.no_grad():
        _ = model.net(X)
    # measure
    times = []
    with torch.no_grad():
        for _ in range(n_batches):
            try:
                X, _ = next(it)
            except StopIteration:
                it = iter(ds.val_loader)
                X, _ = next(it)
            X = X.view(X.size(0), -1).float()
            t0 = time.perf_counter()
            _ = model.net(X)
            times.append((time.perf_counter() - t0) * 1e6)
    return sum(times) / len(times)

fp32_val_loss, fp32_acc = eval_loss_acc(fp32_model, ds.val_loader)
ptq_val_loss, ptq_acc = eval_loss_acc(ptq_model, ds.val_loader)
qat_val_loss, qat_acc = eval_loss_acc(qat_model, ds.val_loader)

rows = [
    ("FP32 baseline", fp32_val_loss, fp32_acc, state_size_bytes(fp32_model), avg_latency_us(fp32_model)),
    ("PTQ int8 (weight-only)", ptq_val_loss, ptq_acc, state_size_bytes(ptq_model), avg_latency_us(ptq_model)),
    ("QAT 8da4w (converted)", qat_val_loss, qat_acc, state_size_bytes(qat_model), avg_latency_us(qat_model)),
]

t = PrettyTable()
t.title = "Quantization recipes on MNIST"
t.field_names = ["model", "val loss", "val acc", "state size (KB)", "fwd latency (µs/batch)"]
for name, loss, acc, sz, lat in rows:
    t.add_row([name, f"{loss:.4f}", f"{acc*100:.2f}%", f"{sz/1024:.1f}", f"{lat:.0f}"])
print(t)


+----------------------------------------------------------------------------------------+
|                             Quantization recipes on MNIST                              |
+------------------------+----------+---------+-----------------+------------------------+
|         model          | val loss | val acc | state size (KB) | fwd latency (µs/batch) |
+------------------------+----------+---------+-----------------+------------------------+
|     FP32 baseline      |  2.0587  |  53.48% |      429.3      |          1382          |
| PTQ int8 (weight-only) |  2.0587  |  53.38% |      112.6      |          1721          |
| QAT 8da4w (converted)  |  2.0729  |  44.53% |      406.7      |          3512          |
+------------------------+----------+---------+-----------------+------------------------+


## 6.2 Discussion

The expected pattern at MNIST scale + a short training budget:

- **FP32 baseline** sets the accuracy ceiling and the size floor.
- **PTQ int8** shrinks the state-dict (per-channel int8 weights vs FP32) at a small accuracy hit. Latency may be similar or *slower* on CPU at this tiny scale — the torchao dispatch overhead dominates the math savings.
- **QAT 8da4w** has the smallest converted state-dict (4-bit weights) but the highest accuracy cost since 4-bit is aggressive. With longer training the QAT path closes most of the gap to FP32 — at the short budget used here the recovery is partial.

The headline pedagogical point: **quantization is a real Pareto trade-off**. The right point on the curve depends on deployment constraints (memory? latency? accuracy floor?). PTQ is the cheap default; QAT is the recourse when PTQ accuracy isn't acceptable.
